# 02 — GEE AlphaEarth `Sample Augmentation`

This notebook demonstrates training **`Sample Augmentation`** using Google's Annual Satellite Embedding dataset (`GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL`) via `lc_framework.samples`:

1. Configure per-class cosine-similarity thresholds keyed by `Class Schema` IDs or canonical names.
2. Build the AlphaEarth annual mosaic and compute per-class prototype cosine similarity (`sim_<id>`).
3. Threshold, vectorize, and deduplicate sites to centroids tagged with `class_id`, `class_name`, `confidence`, and `sample_source="alphaearth_similarity"`.
4. Optionally generate the **explicitly uncalibrated** `DiagnosticSoftmaxOverlay` (`qa_softmax_<id>`) for visual QA (never confused with a calibrated `Class Probability Stack`).
5. Export augmented points to GEE Assets or Google Drive for visual QA before merging into training sets.

In [ ]:
# 1. Colab Bootstrap — install [gee] extra for GEE AlphaEarth Sample Augmentation
# Replace <your-org> with your company GitHub organization
# !pip install -q "lc-framework[gee] @ git+https://github.com/<your-org>/LC_Framework.git"

from pathlib import Path
from lc_framework.config import ClassDefinition, ClassSchema, ProjectConfig
from lc_framework.samples import (
    SampleAugmentationParams,
    export_augmented_samples_to_asset,
    export_augmented_samples_to_drive,
    merge_augmented_samples,
    resolve_class_thresholds,
    run_sample_augmentation,
)

## 2. Load `Class Schema` & Per-Class Similarity Thresholds (Self-Contained Fallback)
Works cleanly in Google Colab even without cloning the repository.

In [ ]:
# Uncomment in Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')

CONFIG_PATH = Path("../configs/example_project.yaml")
if CONFIG_PATH.exists():
    cfg = ProjectConfig.from_yaml(CONFIG_PATH)
    schema = cfg.class_schema
else:
    cfg = None
    schema = ClassSchema(
        classes=(
            ClassDefinition(id=1, name="Forest", color="#1b5e20"),
            ClassDefinition(id=2, name="Cropland", color="#fbc02d"),
            ClassDefinition(id=5, name="Built-up", color="#e53935"),
        )
    )

# Tune similarity thresholds by class name or integer ID
PARAMS = SampleAugmentationParams(
    year=2024,
    scale_m=cfg.scale_m if cfg is not None else 10.0,
    default_threshold=0.90,
    eight_connected=False,
)
AOI_ASSET_ID = (
    cfg.aoi_asset
    if cfg is not None and cfg.aoi_asset
    else "projects/your-project/assets/aoi"
)
THRESHOLDS = resolve_class_thresholds(
    schema,
    thresholds={"Forest": 0.92, "Cropland": 0.88},
    default_threshold=PARAMS.default_threshold,
)
print("Class Schema:", [(c.id, c.name) for c in schema.classes])
print("Resolved thresholds:", THRESHOLDS)

## 3. Run AlphaEarth `Sample Augmentation` in Google Earth Engine
Flip `RUN_GEE_AUGMENTATION = True` after running `ee.Authenticate()` and `ee.Initialize(project=...)` in Colab.

In [ ]:
RUN_GEE_AUGMENTATION = False

if RUN_GEE_AUGMENTATION:
    import ee
    # ee.Authenticate()
    # ee.Initialize(project="your-gcp-project")

    training_fc = ee.FeatureCollection("projects/your-project/assets/reference_points")
    aoi = ee.FeatureCollection(AOI_ASSET_ID).geometry()

    aug_res = run_sample_augmentation(
        training_fc,
        config=cfg,
        class_schema=schema if cfg is None else None,
        region=aoi,
        thresholds=THRESHOLDS,
        params=PARAMS,
        class_property="class_id",
        include_diagnostic_overlay=True,
        temperature=10.0,
    )
    if aug_res.diagnostic_overlay is not None:
        print("Overlay calibrated?", aug_res.diagnostic_overlay.calibrated)
        print("Overlay bands:", aug_res.diagnostic_overlay.band_names)
        print("Note:", aug_res.diagnostic_overlay.disclaimer)

    # Export augmented candidates first (recommended for large AOIs)
    export_augmented_samples_to_asset(
        aug_res.candidate_points,
        asset_id="projects/your-project/assets/alphaearth_candidates_2024",
        dry_run=True,  # Set False to launch export task
    )
    export_augmented_samples_to_drive(
        aug_res.candidate_points,
        description="alphaearth_candidates_2024",
        folder="LC_Framework_Exports",
        dry_run=True,
    )
    # After visual QA, merge with original reference samples (keep validation split pure!)
    merged_fc = merge_augmented_samples(training_fc, aug_res.candidate_points)
    print("Prepared GEE augmentation pipeline ->", type(merged_fc).__name__)
else:
    print("GEE execution skipped (RUN_GEE_AUGMENTATION=False). Threshold & schema checks passed.")